# 09 — Exploration de la recherche plein texte (futur outil de l'agent)

**Objectif** : voir comment PostgreSQL découpe et retrouve **les vrais termes** avant d'en faire un outil `rechercher_mots_cles` pour l'agent. **Aucun appel au LLM : ce notebook est gratuit.**

1. connexion ; extension `unaccent` et configuration `fr_sans_accents` ;
2. le découpage d'un texte en lexèmes : accents, pluriels, majuscules, codes, pouces ;
3. la traduction d'une requête (`websearch_to_tsquery`) : ET implicite, expressions exactes, `or`, exclusion ;
4. la recherche sur le vrai corpus, avec les mots trouvés surlignés ;
5. une batterie de requêtes de type « agent » : configuration française standard contre sans accents ;
6. une phrase complète contre des mots-clés ;
7. les mots les plus fréquents du corpus (ceux qui font du bruit) ;
8. la complémentarité avec la recherche dense ;
9. la latence.

Le texte est vectorisé **à la volée** dans les requêtes (207 passages : quelques millisecondes). La colonne indexée viendra avec le code propre, une fois la configuration choisie.

## 1. Connexion et configuration de recherche

Trois objets sont créés dans ta base de dev (sans effet sur la table) :

- l'extension **`unaccent`**, qui retire les accents (« dégâts » -> « degats ») ;
- le dictionnaire **`fr_mots_vides`**, qui écarte les mots vides français (le, de, à…) ;
- la configuration **`fr_sans_accents`** : une copie de la configuration française où chaque mot traverse une **chaîne de dictionnaires**, dans cet ordre :
  1. `fr_mots_vides` : un mot vide est écarté ; sinon, il passe au suivant ;
  2. `unaccent` : les accents sont retirés ;
  3. `french_stem` : racinisation française.

**Pourquoi cet ordre** : si `unaccent` passait en premier, « à » deviendrait « a », qui n'est plus reconnu comme mot vide, et se retrouverait dans un passage sur trois.

C'est cette configuration, et non un appel à `unaccent()`, qui pourra servir dans une colonne générée : Postgres n'y accepte que des fonctions garanties invariables, et `unaccent()` seule ne l'est pas.

In [2]:
import time

import pandas as pd
import psycopg
from dotenv import load_dotenv

from assistant_regles.ingest.config import trouver_racine
from assistant_regles.rag.store import TABLE

load_dotenv(trouver_racine() / ".env")
conn = psycopg.connect(autocommit=True)

conn.execute("CREATE EXTENSION IF NOT EXISTS unaccent")
conn.execute("""
DO $$
BEGIN
    IF NOT EXISTS (SELECT 1 FROM pg_ts_dict WHERE dictname = 'fr_mots_vides') THEN
        CREATE TEXT SEARCH DICTIONARY fr_mots_vides (TEMPLATE = simple, STOPWORDS = french, ACCEPT = false);
    END IF;
    IF NOT EXISTS (SELECT 1 FROM pg_ts_config WHERE cfgname = 'fr_sans_accents') THEN
        CREATE TEXT SEARCH CONFIGURATION fr_sans_accents (COPY = french);
        ALTER TEXT SEARCH CONFIGURATION fr_sans_accents
            ALTER MAPPING FOR hword, hword_part, word WITH fr_mots_vides, unaccent, french_stem;
    END IF;
END $$;
""")
CONFIGS = ["simple", "french", "fr_sans_accents"]
print("configurations :", CONFIGS)

configurations : ['simple', 'french', 'fr_sans_accents']


## 2. Découpage d'un texte en lexèmes

`to_tsvector` transforme un texte en **lexèmes** (racines normalisées, avec leur position). Une recherche trouve un passage si les lexèmes de la requête y figurent. À observer, configuration par configuration :

- **accents** : « dégâts » et « degats » donnent-ils le même lexème ?
- **pluriels et conjugaisons** : « tir », « tirs », « tirer » ;
- **majuscules** : les mots-clés du livre sont mis en minuscules ;
- **codes** (« 03.01 ») et **distances** (« 6" ») ;
- **mots composés** (« Grande-Cible ») et **apostrophes** (« l'unité ») ;
- **mots vides** (le, les, de…), normalement retirés.

`simple` sert de témoin : ni racinisation, ni mots vides.

In [3]:
PHRASES = [
    "Les unités subissent des dégâts",
    "degats DÉGÂTS Dégâts",
    "tir tirs tirer tiré TIRER",
    "l'unité TIRE avec un PISTOLET à 6\" (voir 03.01)",
    "Grande-Cible, demi-effectif, Points de Commandement",
    "à la fin de la phase de Mouvement",
]


def lexemes(texte, config):
    return conn.execute("SELECT to_tsvector(%s::regconfig, %s)::text", (config, texte)).fetchone()[0]


pd.set_option("display.max_colwidth", 200)
pd.DataFrame([{"texte": p, **{c: lexemes(p, c) for c in CONFIGS}} for p in PHRASES]).set_index("texte")

,simple,french,fr_sans_accents
texte,,,
Les unités subissent des dégâts,'des':4 'dégâts':5 'les':1 'subissent':3 'unités':2,'dégât':5 'le':1 'sub':3 'unit':2,'degat':5 'le':1 'sub':3 'unit':2
degats DÉGÂTS Dégâts,"'degats':1 'dégâts':2,3","'degat':1 'dégât':2,3","'degat':1,2,3"
tir tirs tirer tiré TIRER,"'tir':1 'tirer':3,5 'tirs':2 'tiré':4","'tir':1,2,3,4,5","'tir':1,2,3,4,5"
"l'unité TIRE avec un PISTOLET à 6"" (voir 03.01)",'03.01':10 '6':8 'avec':4 'l':1 'pistolet':6 'tire':3 'un':5 'unité':2 'voir':9 'à':7,'03.01':10 '6':8 'pistolet':6 'tir':3 'unit':2 'voir':9,'03.01':10 '6':8 'pistolet':6 'tir':3 'unit':2 'voir':9
"Grande-Cible, demi-effectif, Points de Commandement",'cible':3 'commandement':9 'de':8 'demi':5 'demi-effectif':4 'effectif':6 'grande':2 'grande-cible':1 'points':7,'cibl':3 'command':9 'dem':5 'demi-effect':4 'effect':6 'grand':2 'grande-cibl':1 'point':7,'cibl':3 'command':9 'dem':5 'demi-effect':4 'effect':6 'grand':2 'grande-cibl':1 'point':7
à la fin de la phase de Mouvement,"'de':4,7 'fin':3 'la':2,5 'mouvement':8 'phase':6 'à':1",'fin':3 'mouv':8 'phas':6,'fin':3 'mouv':8 'phas':6


## 3. Traduction d'une requête : `websearch_to_tsquery`

C'est la fonction que l'outil utilisera. Elle comprend la syntaxe des moteurs de recherche web, que Claude connaît bien, et **ne lève jamais d'erreur de syntaxe** :

| syntaxe | sens | tsquery |
|---|---|---|
| `pistolet corps` | les deux mots (**ET** implicite) | `&` |
| `"demi effectif"` | expression exacte, mots consécutifs | `<->` |
| `avancer or retraite` | l'un ou l'autre | `\|` |
| `-véhicule` | exclure | `!` |

Le **ET implicite** est la raison pour laquelle l'agent devra écrire **quelques mots-clés** et non une phrase : chaque mot ajouté est une contrainte de plus.

In [4]:
REQUETES_SYNTAXE = [
    "pistolet corps",
    "degats pistolets",
    '"demi effectif"',
    "avancer or retraite",
    "ébranlé -véhicule",
    "est-ce que je peux tirer ?",
]
pd.DataFrame([
    {"requête": r, **{c: conn.execute("SELECT websearch_to_tsquery(%s::regconfig, %s)::text", (c, r)).fetchone()[0]
                      for c in CONFIGS}}
    for r in REQUETES_SYNTAXE
]).set_index("requête")

,simple,french,fr_sans_accents
requête,,,
pistolet corps,'pistolet' & 'corps','pistolet' & 'corp','pistolet' & 'corp'
degats pistolets,'degats' & 'pistolets','degat' & 'pistolet','degat' & 'pistolet'
"""demi effectif""",'demi' <-> 'effectif','dem' <-> 'effect','dem' <-> 'effect'
avancer or retraite,'avancer' | 'retraite','avanc' | 'retrait','avanc' | 'retrait'
ébranlé -véhicule,'ébranlé' & !'véhicule','ébranl' & !'véhicul','ebranl' & !'vehicul'
est-ce que je peux tirer ?,'est-ce' <-> 'est' <-> 'ce' & 'que' & 'je' & 'peux' & 'tirer','est-ce' & 'peux' & 'tir','est-ce' & 'peux' & 'tir'


## 4. Recherche sur le vrai corpus

Classement par **`ts_rank_cd`** : il tient compte de la fréquence des mots trouvés **et de leur proximité** dans le texte. `ts_headline` montre les extraits où les mots ont été trouvés, entre « ».

In [5]:
SQL_TEXTE = f"""
WITH q AS (SELECT websearch_to_tsquery(%(config)s::regconfig, %(requete)s) AS requete)
SELECT code, page_debut,
       ts_rank_cd(to_tsvector(%(config)s::regconfig, texte), q.requete) AS rang,
       ts_headline(%(config)s::regconfig, texte, q.requete,
                   'StartSel=«, StopSel=», MaxFragments=2, MaxWords=12, MinWords=4') AS extrait
FROM {TABLE}, q
WHERE to_tsvector(%(config)s::regconfig, texte) @@ q.requete
ORDER BY rang DESC
LIMIT %(k)s
"""


def rechercher_texte(requete, k=5, config="fr_sans_accents"):
    lignes = conn.execute(SQL_TEXTE, {"config": config, "requete": requete, "k": k}).fetchall()
    return pd.DataFrame(lignes, columns=["code", "page", "rang", "extrait"])


rechercher_texte("DEMI-EFFECTIF")

,code,page,rang,extrait
0,08.05,30,0.5,"donc ni à «demi»-«effectif» ni en dessous de son «demi»-«effectif» ... figurines, elle est donc à «demi»-«effectif» et on doit faire"
1,NaN,86,0.5,«effectif» initial. |\n| À «DEMI»-«EFFECTIF» | Les points de vie restants ... «effectif» initial. |\n| EN DESSOUS DU «DEMI»-«EFFECTIF» | Les points de vie restants
2,NaN,86,0.2,appui.\n\n### «EFFECTIF» INITIAL ET «DEMI»-«EFFECTIF»\nLe nombre de figurines ... dessous de leur - «demi»-«effectif». Le sens de ces termes varie selon
3,NaN,86,0.2,PEUVENT PAS ÊTRE À «DEMI»-«EFFECTIF»\nSi la caractéristique ... être en dessous de son «demi»-«effectif» comme décrit à droite).\n\n### AJOUTER
4,08.03,30,0.1,"ébranlée.\n- L'unité est à «demi»-«effectif», ou en dessous"


## 5. Batterie de requêtes de type « agent »

**Remplace la liste par des termes réels du livre** : mots-clés en majuscules, noms d'armes ou d'aptitudes, noms de stratagèmes, expressions exactes, et quelques variantes **sans accents** ou **au pluriel**, comme un joueur les taperait.

Pour chaque requête : le nombre de passages trouvés et les premiers codes, en configuration française standard puis sans accents. Une ligne à 0 dans une seule des deux configurations révèle une différence de normalisation à comprendre.

In [21]:
REQUETES = [
    "DEMI-EFFECTIF",
    "demi effectif",
    "BLESSURES DÉVASTATRICES",
    "ATTAQUES BONUS",
    "[ATTAQUES BONUS]",
    "PISTOLET",
    "JETS DE SAUVEGARDE",
    "ligne de vue",
    "pistolets",
    "dégâts mortels",
    "test d'ébranlement",
    "points de commandement",
    "battre en retraite",
    "retraite",
    "avancer or advance",
    "03.03",
]


def compter(requete, config):
    return conn.execute(
        f"SELECT count(*) FROM {TABLE} WHERE to_tsvector(%(c)s::regconfig, texte) @@ websearch_to_tsquery(%(c)s::regconfig, %(r)s)",
        {"c": config, "r": requete},
    ).fetchone()[0]


lignes = []
for r in REQUETES:
    ligne = {"requête": r}
    for c in ("french", "fr_sans_accents"):
        ligne[f"n {c}"] = compter(r, c)
        ligne[f"top {c}"] = list(rechercher_texte(r, k=5, config=c)["code"])
    lignes.append(ligne)
pd.DataFrame(lignes).set_index("requête")

,n french,top french,n fr_sans_accents,top fr_sans_accents
requête,,,,
DEMI-EFFECTIF,6,"[08.05, nan, nan, nan, 08.03]",6,"[08.05, nan, nan, nan, 08.03]"
demi effectif,6,"[08.05, nan, nan, nan, nan]",6,"[08.05, nan, nan, nan, nan]"
BLESSURES DÉVASTATRICES,3,"[24.10, nan, 24.23]",3,"[24.10, nan, 24.23]"
ATTAQUES BONUS,1,[24.11],1,[24.11]
[ATTAQUES BONUS],1,[24.11],1,[24.11]
PISTOLET,5,"[24.27, 05.04, 05.04, 05.04, 05.04]",5,"[24.27, 05.04, 05.04, 05.04, 05.04]"
JETS DE SAUVEGARDE,8,"[05.03, 15.02, 05.04, 05.04, 05.04]",8,"[05.03, 15.02, 05.04, 05.04, 05.04]"
ligne de vue,5,"[06.01, 13.11, 13.05, 13.10, 13.11]",5,"[06.01, 13.11, 13.05, 13.10, 13.11]"
pistolets,5,"[24.27, 05.04, 05.04, 05.04, 05.04]",5,"[24.27, 05.04, 05.04, 05.04, 05.04]"


In [16]:
# Détail d'une requête : extraits mis en avant
rechercher_texte(REQUETES[0], k=8)

,code,page,rang,extrait
0,08.05,30,0.5,"donc ni à «demi»-«effectif» ni en dessous de son «demi»-«effectif» ... figurines, elle est donc à «demi»-«effectif» et on doit faire"
1,NaN,86,0.5,«effectif» initial. |\n| À «DEMI»-«EFFECTIF» | Les points de vie restants ... «effectif» initial. |\n| EN DESSOUS DU «DEMI»-«EFFECTIF» | Les points de vie restants
2,NaN,86,0.2,appui.\n\n### «EFFECTIF» INITIAL ET «DEMI»-«EFFECTIF»\nLe nombre de figurines ... dessous de leur - «demi»-«effectif». Le sens de ces termes varie selon
3,NaN,86,0.2,PEUVENT PAS ÊTRE À «DEMI»-«EFFECTIF»\nSi la caractéristique ... être en dessous de son «demi»-«effectif» comme décrit à droite).\n\n### AJOUTER
4,08.03,30,0.1,"ébranlée.\n- L'unité est à «demi»-«effectif», ou en dessous"
5,NaN,87,0.1,figurines.\n\n### «EFFECTIF» INITIAL ET «DEMI»-«EFFECTIF»\n►Récupérer des Points de Vie Perdus


## 6. Phrase complète contre mots-clés

La même intention, formulée comme un joueur puis comme des mots-clés. Avec le ET implicite, la phrase complète exige que **tous** ses mots significatifs soient présents dans le même passage : on s'attend à peu ou pas de résultats. C'est ce que la description de l'outil devra dire à l'agent.

In [17]:
PAIRES = [
    ("Est-ce que mon unité peut tirer après avoir avancé ?", "AVANCER tirer"),
    ("Combien de points de commandement je gagne par round ?", '"points de commandement"'),
    ("Que se passe-t-il si mon unité est en dessous de son demi-effectif ?", '"demi effectif"'),
]
pd.DataFrame([
    {"phrase": p, "n phrase": compter(p, "fr_sans_accents"), "mots-clés": m, "n mots-clés": compter(m, "fr_sans_accents")}
    for p, m in PAIRES
])

,phrase,n phrase,mots-clés,n mots-clés
0,Est-ce que mon unité peut tirer après avoir avancé ?,0,AVANCER tirer,6
1,Combien de points de commandement je gagne par round ?,0,"""points de commandement""",3
2,Que se passe-t-il si mon unité est en dessous de son demi-effectif ?,0,"""demi effectif""",6


## 7. Les mots les plus fréquents du corpus

`ts_stat` compte, pour chaque lexème, le nombre de passages qui le contiennent (`ndoc`). Un mot présent dans une grande partie du corpus (« unit », « figurin »…) ne discrimine presque rien : seul dans une requête, il renvoie du bruit. C'est une indication utile pour la description de l'outil : privilégier les termes **rares et spécifiques**.

In [18]:
stats = conn.execute(f"""
    SELECT word AS lexeme, ndoc, round(100.0 * ndoc / (SELECT count(*) FROM {TABLE}), 1) AS pct_passages
    FROM ts_stat($$SELECT to_tsvector('fr_sans_accents', texte) FROM {TABLE}$$)
    ORDER BY ndoc DESC LIMIT 30
""").fetchall()
pd.DataFrame(stats, columns=["lexème", "passages", "% des passages"])

,lexème,passages,% des passages
0,le,172,83.1
1,unit,162,78.3
2,cet,124,59.9
3,bataill,122,58.9
4,regl,118,57.0
5,figurin,118,57.0
6,si,104,50.2
7,chaqu,99,47.8
8,attaqu,82,39.6
9,a,80,38.6


## 8. Complémentarité avec la recherche dense

Pour chaque requête de la section 5 : les 5 premiers codes de la recherche **dense** (celle d'aujourd'hui) et de la recherche **plein texte**. Ce qui compte, ce sont les passages trouvés **seulement** par le plein texte : c'est ce que l'outil apportera à l'agent. Si cette colonne est presque toujours vide, l'outil n'apporte rien.

Cette section charge BGE-M3 (quelques secondes).

In [22]:
REQUETES

['DEMI-EFFECTIF',
 'demi effectif',
 'BLESSURES DÉVASTATRICES',
 'ATTAQUES BONUS',
 '[ATTAQUES BONUS]',
 'PISTOLET',
 'JETS DE SAUVEGARDE',
 'ligne de vue',
 'pistolets',
 'dégâts mortels',
 "test d'ébranlement",
 'points de commandement',
 'battre en retraite',
 'retraite',
 'avancer or advance',
 '03.03']

In [23]:
from assistant_regles.rag.config import charger_config
from assistant_regles.rag.embeddings import EncodeurBGEM3
from assistant_regles.rag.recherche import MoteurRecherche

moteur = MoteurRecherche(EncodeurBGEM3.charger(charger_config().embeddings), conn)

lignes = []
for r in REQUETES:
    dense = [x.code for x in moteur.rechercher(r, 5)]
    texte = list(rechercher_texte(r, k=5)["code"])
    lignes.append({"requête": r, "dense": dense, "plein texte": texte,
                   "seulement plein texte": [c for c in texte if c not in dense]})
pd.DataFrame(lignes).set_index("requête")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

,dense,plein texte,seulement plein texte
requête,,,
DEMI-EFFECTIF,"[None, None, 05.04, None, 05.04]","[08.05, nan, nan, nan, 08.03]","[08.05, nan, nan, nan, 08.03]"
demi effectif,"[None, 15.05, 15.06, 24.05, None]","[08.05, nan, nan, nan, nan]","[08.05, nan, nan, nan, nan]"
BLESSURES DÉVASTATRICES,"[24.10, 06.02, 24.08, 05.04, 05.04]","[24.10, nan, 24.23]","[nan, 24.23]"
ATTAQUES BONUS,"[24.11, 24.05, 13.08, 19.02, 24.28]",[24.11],[]
[ATTAQUES BONUS],"[24.11, 24.05, 04.03, 05.04, 19.02]",[24.11],[]
PISTOLET,"[24.27, 04.02, 02.04, 24.07, 24.04]","[24.27, 05.04, 05.04, 05.04, 05.04]","[05.04, 05.04, 05.04, 05.04]"
JETS DE SAUVEGARDE,"[05.03, 05.04, 06.03, 05.04, 05.04]","[05.03, 15.02, 05.04, 05.04, 05.04]",[15.02]
ligne de vue,"[06.01, 13.11, 06.03, 13.11, 10.07]","[06.01, 13.11, 13.05, 13.10, 13.11]","[13.05, 13.10]"
pistolets,"[24.27, 24.04, 24.07, 02.04, 24.37]","[24.27, 05.04, 05.04, 05.04, 05.04]","[05.04, 05.04, 05.04, 05.04]"


## 9. Latence

Recherche plein texte **sans index** (vectorisation à la volée des 207 passages), pour avoir un ordre de grandeur. Avec une colonne générée et un index GIN, ce sera encore plus rapide ; dans tous les cas, c'est négligeable à côté d'un appel au LLM.

In [14]:
debut = time.perf_counter()
for _ in range(20):
    rechercher_texte("pistolet corps à corps")
print(f"{(time.perf_counter() - debut) / 20 * 1000:.1f} ms par recherche (sans index)")

14.0 ms par recherche (sans index)


## Bilan : décisions pour l'outil `rechercher_mots_cles`

- **configuration** : `fr_sans_accents` retrouve-t-elle bien les variantes avec et sans accents, sans effet indésirable ?
- **termes piégeux** : codes, distances, mots composés, expressions ; à signaler dans la description de l'outil ?
- **consignes à l'agent** : quelques mots-clés **rares**, expressions exactes entre guillemets, `or` pour les variantes ;
- **complémentarité** : le plein texte trouve-t-il des passages que la recherche dense rate ? C'est la condition pour que l'outil vaille la peine.

In [ ]:
conn.close()